# Semantic Hybrid Recommender on Amazon Reviews

This notebook builds a staged recommendation experiment for Amazon Reviews:

```text
Popularity baseline
→ ID-based Two-Tower
→ Mini-DLRM-style ranking model
→ Hybrid Mini-DLRM + frozen semantic item embeddings
```

This project implements a compact DLRM-inspired ranking model for educational and experimental purposes, not a full-scale production DLRM system.

## Retrieval vs Ranking Clarification

This notebook evaluates models on sampled candidate sets using leave-one-out ranking. It does not implement full-scale industrial ANN retrieval over millions of items. The goal is to compare collaborative, feature-interaction, and semantic-hybrid ranking signals.

## Cold-start and Long-tail Motivation

Collaborative embeddings require user-item interactions to become meaningful. Long-tail or new items often have too few interactions to learn stable embeddings. A semantic encoder can place items into a meaningful content space based on title, category, and description before sufficient user feedback exists.

## Setup

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch

from src.data import (
    apply_id_mappings,
    build_category_features,
    build_dense_features,
    build_id_mappings,
    cap_interactions,
    filter_interactions,
    generate_synthetic_amazon_data,
    load_metadata,
    load_reviews,
    normalize_columns,
    temporal_split,
)
from src.metrics import evaluate_leave_one_out, metrics_to_frame
from src.models import HybridDLRM, MiniDLRM, TwoTower
from src.negative_sampling import build_eval_candidates, build_training_samples, build_user_positive_items
from src.semantic_embeddings import build_item_text, load_or_create_semantic_embeddings
from src.train import ItemFeatureStore, make_popularity_scorer, make_torch_scorer, recommend_top_k, train_model
from src.utils import get_device, seed_everything

seed_everything(42)
device = get_device()
device

In [ ]:
# Larger sparse long-tail real-data experiment configuration.
# Keep paths as None for synthetic fallback, or point them at local Amazon Reviews files.
RUN_NAME = "movies_tv_long_tail_minilm"
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REVIEWS_PATH = "data/raw/movies_tv_reviews_subset.jsonl"
METADATA_PATH = "data/raw/movies_tv_metadata_subset.jsonl"
MAX_INTERACTIONS = 50_000
SEMANTIC_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPOCHS = 5
RUN_SEMANTIC_VIS = False
SEMANTIC_VIS_METHOD = "pca"  # Use "tsne" for an optional slower visualization.

# Use SEMANTIC_MODEL_NAME = "tfidf-svd" for offline-only smoke runs.

# Model and evaluation defaults.
EMB_DIM = 32
BATCH_SIZE = 1024
LR = 1e-3
TRAIN_NEGATIVES = 4
EVAL_NEGATIVES = 99
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 2
K = 10
METRICS_OUT = REPO_ROOT / "results" / f"metrics_{RUN_NAME}_{RUN_TIMESTAMP}.csv"
LONG_TAIL_OUT = REPO_ROOT / "results" / f"long_tail_{RUN_NAME}_{RUN_TIMESTAMP}.csv"

## Load Data or Synthetic Fallback

In [ ]:
reviews_path = (REPO_ROOT / REVIEWS_PATH) if REVIEWS_PATH else None
metadata_path = (REPO_ROOT / METADATA_PATH) if METADATA_PATH else None

using_synthetic = False
if reviews_path and reviews_path.exists():
    reviews_raw = load_reviews(str(reviews_path))
    metadata_raw = load_metadata(str(metadata_path)) if metadata_path else None
else:
    using_synthetic = True
    print("Using synthetic demo dataset. For real evaluation, provide Amazon Reviews files.")
    reviews_raw, metadata_raw = generate_synthetic_amazon_data()

ACTIVE_SEMANTIC_MODEL_NAME = SEMANTIC_MODEL_NAME
if using_synthetic and SEMANTIC_MODEL_NAME != "tfidf-svd":
    print("Synthetic fallback active; using TF-IDF + SVD embeddings to keep the demo offline-safe.")
    ACTIVE_SEMANTIC_MODEL_NAME = "tfidf-svd"

reviews_raw.head(), metadata_raw.head() if metadata_raw is not None else None

## Preprocess Data

In [ ]:
reviews, metadata = normalize_columns(reviews_raw, metadata_raw)
reviews = cap_interactions(reviews, MAX_INTERACTIONS, seed=42)
interactions = filter_interactions(
    reviews,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS,
)
num_stat_users = interactions["user_id"].nunique()
num_stat_items = interactions["item_id"].nunique()
num_stat_interactions = len(interactions)
sparsity = 1.0 - (num_stat_interactions / max(1, num_stat_users * num_stat_items))
dataset_stats = pd.DataFrame(
    [{"users": num_stat_users, "items": num_stat_items, "interactions": num_stat_interactions, "sparsity": sparsity}]
)
experiment_config = {
    "run_name": RUN_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "dataset": "synthetic" if using_synthetic else "Movies_and_TV",
    "users": num_stat_users,
    "items": num_stat_items,
    "interactions": num_stat_interactions,
    "sparsity": sparsity,
    "max_interactions": MAX_INTERACTIONS,
    "epochs": EPOCHS,
    "min_user_interactions": MIN_USER_INTERACTIONS,
    "min_item_interactions": MIN_ITEM_INTERACTIONS,
    "semantic_model": ACTIVE_SEMANTIC_MODEL_NAME,
    "train_negatives": TRAIN_NEGATIVES,
    "eval_negatives": EVAL_NEGATIVES,
    "k": K,
}
display(dataset_stats)
interactions.head()

## Train/Validation/Test Split

In [ ]:
train_df, val_df, test_df = temporal_split(interactions)
user_mapping, item_mapping = build_id_mappings(train_df, val_df, test_df)
train_df = apply_id_mappings(train_df, user_mapping, item_mapping)
val_df = apply_id_mappings(val_df, user_mapping, item_mapping)
test_df = apply_id_mappings(test_df, user_mapping, item_mapping)

num_users = len(user_mapping)
num_items = len(item_mapping)
print(train_df.shape, val_df.shape, test_df.shape, num_users, num_items)

## Negative Sampling

This project uses random negative sampling for simplicity. Future work can use hard negatives or popularity-aware negative sampling.

In [ ]:
all_positive_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
known_positive_items = build_user_positive_items(all_positive_df)
train_samples = build_training_samples(
    train_df,
    known_positive_items,
    num_items,
    num_negatives=TRAIN_NEGATIVES,
    seed=42,
)
val_candidates = build_eval_candidates(val_df, known_positive_items, num_items, EVAL_NEGATIVES, seed=43)
test_candidates = build_eval_candidates(test_df, known_positive_items, num_items, EVAL_NEGATIVES, seed=44)
train_samples.head(), len(val_candidates), len(test_candidates)

## Metrics

We report Recall@10, HitRate@10, and NDCG@10. For leave-one-out evaluation, Recall@K and HitRate@K are equivalent, but both are reported for readability.

## Popularity Baseline

In [ ]:
results = {}
model_scorers = {}
popularity_scorer = make_popularity_scorer(train_df, num_items)
results["Popularity"] = evaluate_leave_one_out(None, test_candidates, k=K, scorer=popularity_scorer)
results["Popularity"]

## Two-Tower Baseline

Recommendation is formulated as implicit feedback prediction. Positive samples are observed interactions and negative samples are sampled unobserved items. The model outputs raw logits, and BCEWithLogitsLoss combines sigmoid and binary cross entropy in a numerically stable way.

In [ ]:
two_tower = TwoTower(num_users, num_items, emb_dim=EMB_DIM)
two_tower = train_model(
    two_tower,
    train_samples,
    eval_candidates=val_candidates,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
two_tower_scorer = make_torch_scorer(two_tower, device=device)
model_scorers["Two-Tower"] = two_tower_scorer
results["Two-Tower"] = evaluate_leave_one_out(
    two_tower,
    test_candidates,
    k=K,
    scorer=two_tower_scorer,
)
results["Two-Tower"]

## Mini-DLRM-style Ranking Model

Mini-DLRM learns behavioral similarity from user-item interactions and feature interactions. The model combines user, item, category, and dense item features through pairwise dot-product interactions before ranking candidates.

In [ ]:
category_by_item, category_mapping = build_category_features(item_mapping, metadata)
dense_by_item, dense_feature_names = build_dense_features(item_mapping, metadata)
item_features = ItemFeatureStore(category_by_item=category_by_item, dense_by_item=dense_by_item)
num_categories = len(category_mapping)
dense_dim = dense_by_item.shape[1]
num_categories, dense_dim, dense_feature_names

In [ ]:
mini_dlrm = MiniDLRM(num_users, num_items, num_categories, dense_dim, emb_dim=EMB_DIM)
mini_dlrm = train_model(
    mini_dlrm,
    train_samples,
    eval_candidates=val_candidates,
    item_features=item_features,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
mini_dlrm_scorer = make_torch_scorer(mini_dlrm, item_features=item_features, device=device)
model_scorers["Mini-DLRM"] = mini_dlrm_scorer
results["Mini-DLRM"] = evaluate_leave_one_out(
    mini_dlrm,
    test_candidates,
    k=K,
    scorer=mini_dlrm_scorer,
)
results["Mini-DLRM"]

## Semantic Item Embeddings

The goal is not to fine-tune a text embedding model. Instead, pretrained encoders such as MiniLM, BGE, or e5 are used as frozen semantic feature extractors. This is faster, avoids overfitting on a small recommendation dataset, and preserves pretrained semantic structure.

Embedding generation can be expensive, so semantic vectors are cached and reused across runs.

In [ ]:
item_texts = build_item_text(metadata, item_mapping)
semantic_embeddings = load_or_create_semantic_embeddings(
    item_texts,
    cache_path=REPO_ROOT / "data" / "embeddings" / "semantic_item_embeddings.npy",
    model_name=ACTIVE_SEMANTIC_MODEL_NAME,
)
semantic_dim = semantic_embeddings.shape[1]
semantic_embeddings.shape

## Hybrid Mini-DLRM + Semantic Embeddings

Semantic embeddings may be 384-dimensional, while DLRM-style feature vectors use 32 dimensions. A trainable projection layer maps semantic vectors into the recommender latent space so they can participate in feature interactions.

DLRM-style models learn behavioral similarity from interactions. Semantic encoders learn content similarity from product text. The hybrid model combines both signals, which is especially useful for sparse, long-tail, and cold-start items.

In [ ]:
hybrid = HybridDLRM(
    num_users,
    num_items,
    num_categories,
    dense_dim,
    semantic_embeddings=semantic_embeddings,
    semantic_dim=semantic_dim,
    emb_dim=EMB_DIM,
)
hybrid = train_model(
    hybrid,
    train_samples,
    eval_candidates=val_candidates,
    item_features=item_features,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
hybrid_scorer = make_torch_scorer(hybrid, item_features=item_features, device=device)
model_scorers["Hybrid Mini-DLRM + Semantic"] = hybrid_scorer
results["Hybrid Mini-DLRM + Semantic"] = evaluate_leave_one_out(
    hybrid,
    test_candidates,
    k=K,
    scorer=hybrid_scorer,
)
results["Hybrid Mini-DLRM + Semantic"]

## Evaluation Table

In [ ]:
metrics_df = metrics_to_frame(results)
for key, value in experiment_config.items():
    metrics_df[key] = value
METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(METRICS_OUT, index=False)
print(f"Saved metrics to {METRICS_OUT}")
metrics_df[["Model", f"Recall@{K}", f"HitRate@{K}", f"NDCG@{K}", "items", "interactions", "epochs", "semantic_model"]]

## Ablation Summary

In [ ]:
pd.DataFrame(
    [
        {"Model": "Popularity", "Collaborative Signal": "No", "Feature Interaction": "No", "Semantic Signal": "No"},
        {"Model": "Two-Tower", "Collaborative Signal": "Yes", "Feature Interaction": "No", "Semantic Signal": "No"},
        {"Model": "Mini-DLRM", "Collaborative Signal": "Yes", "Feature Interaction": "Yes", "Semantic Signal": "No"},
        {"Model": "Hybrid Mini-DLRM + Semantic", "Collaborative Signal": "Yes", "Feature Interaction": "Yes", "Semantic Signal": "Yes"},
    ]
)

## Long-tail Analysis

Head items are the top 20% by train popularity. Tail items are the bottom 80%. The table below evaluates Two-Tower, Mini-DLRM, and Hybrid Mini-DLRM + Semantic separately on head and tail test positives. The hypothesis is that the hybrid semantic model should help more on tail items because it can use item content semantics when collaborative interactions are sparse.

In [ ]:
def filter_candidates_by_items(candidates, allowed_items):
    allowed_items = set(allowed_items)
    return [row for row in candidates if row["true_item"] in allowed_items]

popularity = train_df["item_idx"].value_counts()
head_cutoff = max(1, int(np.ceil(num_items * 0.2)))
head_items = set(popularity.sort_values(ascending=False).head(head_cutoff).index.astype(int))
tail_items = set(range(num_items)) - head_items

long_tail_rows = []
for model_name, scorer in model_scorers.items():
    for group_name, item_group in [("head", head_items), ("tail", tail_items)]:
        group_candidates = filter_candidates_by_items(test_candidates, item_group)
        group_metrics = evaluate_leave_one_out(None, group_candidates, k=K, scorer=scorer)
        long_tail_rows.append(
            {
                "Model": model_name,
                "Group": group_name,
                "Test Users": len(group_candidates),
                "Test Items": len({row["true_item"] for row in group_candidates}),
                **group_metrics,
            }
        )

long_tail_df = pd.DataFrame(long_tail_rows)
for key, value in experiment_config.items():
    long_tail_df[key] = value
LONG_TAIL_OUT.parent.mkdir(parents=True, exist_ok=True)
long_tail_df.to_csv(LONG_TAIL_OUT, index=False)
print(f"Saved long-tail metrics to {LONG_TAIL_OUT}")
long_tail_df[["Model", "Group", "Test Users", "Test Items", f"Recall@{K}", f"HitRate@{K}", f"NDCG@{K}"]]

## Example Recommendation Demo

This small product-style demo shows one user's history and top Hybrid recommendations with item metadata snippets.

In [ ]:
idx_to_item = {idx: item_id for item_id, idx in item_mapping.items()}
metadata_lookup = metadata.drop_duplicates("item_id").set_index("item_id") if metadata is not None else pd.DataFrame()

demo_user = int(test_df.iloc[0]["user_idx"])
known_items = known_positive_items.get(demo_user, set())
candidate_items = list(range(num_items))
recommended = recommend_top_k(hybrid_scorer, demo_user, candidate_items, known_items=known_items, k=5)

def describe_item(item_idx):
    item_id = idx_to_item[item_idx]
    if item_id in metadata_lookup.index:
        row = metadata_lookup.loc[item_id]
        return {
            "item_id": item_id,
            "title": row.get("title", ""),
            "category": row.get("main_category", ""),
            "description": str(row.get("description", ""))[:120],
        }
    return {"item_id": item_id, "title": item_id, "category": "", "description": ""}

history = [describe_item(item_idx) for item_idx in list(known_items)[:5]]
recs = [describe_item(item_idx) for item_idx in recommended]
print(f"Demo user_idx: {demo_user}")
print("History sample")
display(pd.DataFrame(history))
print("Hybrid recommendations")
display(pd.DataFrame(recs))

## Optional Semantic Embedding Visualization

If desired, run PCA or t-SNE on `semantic_embeddings` and color by category to inspect whether item content clusters by category. This is intentionally optional and not required for tests.

In [ ]:
if RUN_SEMANTIC_VIS:
    import matplotlib.pyplot as plt

    if SEMANTIC_VIS_METHOD.lower() == "tsne":
        from sklearn.manifold import TSNE

        perplexity = min(30, max(5, (len(semantic_embeddings) - 1) // 3))
        coords = TSNE(n_components=2, perplexity=perplexity, init="pca", learning_rate="auto", random_state=42).fit_transform(
            semantic_embeddings
        )
    else:
        from sklearn.decomposition import PCA

        coords = PCA(n_components=2, random_state=42).fit_transform(semantic_embeddings)

    idx_to_category = {idx: category for category, idx in category_mapping.items()}
    plot_df = pd.DataFrame(
        {
            "x": coords[:, 0],
            "y": coords[:, 1],
            "category": [idx_to_category.get(int(category_idx), "unknown") for category_idx in category_by_item],
        }
    )
    if len(plot_df) > 2_000:
        plot_df = plot_df.sample(2_000, random_state=42)

    plt.figure(figsize=(8, 6))
    top_categories = plot_df["category"].value_counts().head(12).index
    for category, group in plot_df.groupby(plot_df["category"].where(plot_df["category"].isin(top_categories), "other")):
        plt.scatter(group["x"], group["y"], s=14, alpha=0.7, label=str(category)[:35])
    plt.title(f"Semantic item embedding visualization ({SEMANTIC_VIS_METHOD.upper()})")
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.legend(loc="best", fontsize=8)
    plt.show()
else:
    print("Semantic visualization skipped. Set RUN_SEMANTIC_VIS = True to enable PCA/t-SNE.")

## Findings

The completed real-data pass uses an Amazon Reviews 2023 Movies_and_TV subset with 832 users, 2,215 items, 7,356 positive interactions, and 0.9960 sparsity. This setup is still a sampled-candidate ranking experiment, but it is sparse enough to expose differences between ID-only collaborative signals, DLRM-style feature interactions, and semantic item priors.

The earlier tiny dense regime had roughly 47 items. In that setting, semantic embeddings did not help because collaborative repetition was strong and the candidate universe was too small for product-text representations to add much separable signal.

In the larger sparse long-tail regime, Hybrid Mini-DLRM + Semantic outperformed Mini-DLRM overall. It also improved tail-item Recall@10 and NDCG@10 relative to Mini-DLRM, supporting the hypothesis that pretrained semantic item embeddings are more useful when product interactions are sparse but title/category/description metadata remains informative.

## Reporting Visualizations

The cells below reuse saved CSV outputs under `results/` when available. They are reporting-only cells and do not retrain models.

In [ ]:
import matplotlib.pyplot as plt

def latest_csv(pattern):
    matches = sorted((REPO_ROOT / "results").glob(pattern))
    return matches[-1] if matches else None

overall_results_path = latest_csv("metrics_movies_tv_long_tail_minilm_*.csv")
long_tail_results_path = latest_csv("long_tail_movies_tv_long_tail_minilm_*.csv")
ablation_summary_path = latest_csv("ablation_overall_summary_ablation_quick_*.csv")
ablation_tail_summary_path = latest_csv("ablation_long_tail_summary_ablation_quick_*.csv")

report_overall = pd.read_csv(overall_results_path) if overall_results_path else metrics_df.copy()
report_long_tail = pd.read_csv(long_tail_results_path) if long_tail_results_path else long_tail_df.copy()
report_ablation = pd.read_csv(ablation_summary_path) if ablation_summary_path else pd.DataFrame()
report_ablation_tail = pd.read_csv(ablation_tail_summary_path) if ablation_tail_summary_path else pd.DataFrame()

print("Loaded reporting files:")
print("overall:", overall_results_path)
print("long_tail:", long_tail_results_path)
print("ablation_summary:", ablation_summary_path)
print("ablation_tail_summary:", ablation_tail_summary_path)

In [ ]:
training_history = pd.DataFrame(
    [
        {"Model": "Two-Tower", "epoch": 1, "Recall@10": 0.0901, "NDCG@10": 0.0414},
        {"Model": "Two-Tower", "epoch": 2, "Recall@10": 0.0877, "NDCG@10": 0.0402},
        {"Model": "Two-Tower", "epoch": 3, "Recall@10": 0.0938, "NDCG@10": 0.0427},
        {"Model": "Two-Tower", "epoch": 4, "Recall@10": 0.0889, "NDCG@10": 0.0417},
        {"Model": "Two-Tower", "epoch": 5, "Recall@10": 0.0962, "NDCG@10": 0.0449},
        {"Model": "Mini-DLRM", "epoch": 1, "Recall@10": 0.1130, "NDCG@10": 0.0561},
        {"Model": "Mini-DLRM", "epoch": 2, "Recall@10": 0.1947, "NDCG@10": 0.0990},
        {"Model": "Mini-DLRM", "epoch": 3, "Recall@10": 0.1911, "NDCG@10": 0.1016},
        {"Model": "Mini-DLRM", "epoch": 4, "Recall@10": 0.1911, "NDCG@10": 0.1042},
        {"Model": "Mini-DLRM", "epoch": 5, "Recall@10": 0.2043, "NDCG@10": 0.1156},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epoch": 1, "Recall@10": 0.1322, "NDCG@10": 0.0658},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epoch": 2, "Recall@10": 0.1875, "NDCG@10": 0.0944},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epoch": 3, "Recall@10": 0.1959, "NDCG@10": 0.1004},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epoch": 4, "Recall@10": 0.2308, "NDCG@10": 0.1286},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epoch": 5, "Recall@10": 0.2356, "NDCG@10": 0.1357},
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for model_name, group in training_history.groupby("Model"):
    axes[0].plot(group["epoch"], group["Recall@10"], marker="o", label=model_name)
    axes[1].plot(group["epoch"], group["NDCG@10"], marker="o", label=model_name)
axes[0].set_title("Validation Recall@10 by epoch")
axes[1].set_title("Validation NDCG@10 by epoch")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Recall@10")
axes[1].set_ylabel("NDCG@10")
axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

overall_plot = report_overall.set_index("Model")[["Recall@10", "NDCG@10"]]
overall_plot.plot(kind="bar", ax=axes[0], rot=25)
axes[0].set_title("Overall model comparison")
axes[0].set_ylabel("score")
axes[0].grid(axis="y", alpha=0.25)

tail_models = ["Two-Tower", "Mini-DLRM", "Hybrid Mini-DLRM + Semantic"]
head_tail_plot = report_long_tail[report_long_tail["Model"].isin(tail_models)]
head_tail_pivot = head_tail_plot.pivot(index="Model", columns="Group", values="Recall@10")
head_tail_pivot[[col for col in ["head", "tail"] if col in head_tail_pivot.columns]].plot(kind="bar", ax=axes[1], rot=25)
axes[1].set_title("Head vs tail Recall@10")
axes[1].set_ylabel("Recall@10")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
if not report_ablation.empty:
    ablation_plot = report_ablation[
        (report_ablation["epochs"] == 5)
        & (report_ablation["lr"] == 1e-3)
        & (report_ablation["emb_dim"] == 32)
        & (report_ablation["train_negatives"] == 4)
        & (report_ablation["Model"].isin(["Mini-DLRM", "Hybrid Mini-DLRM + Semantic"]))
    ].copy()
    if not ablation_plot.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(
            ablation_plot["Model"],
            ablation_plot["Recall@10_mean"],
            yerr=ablation_plot["Recall@10_std"],
            capsize=5,
            color=["#2563EB" if "Hybrid" in model_name else "#6B7280" for model_name in ablation_plot["Model"]],
        )
        ax.set_title("Mean/std Recall@10 across seeds")
        ax.set_ylabel("Recall@10")
        ax.grid(axis="y", alpha=0.25)
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
    display(ablation_plot)
else:
    print("No ablation summary CSV found. Run scripts/run_ablation.py --quick to generate one.")

## Hyperparameter Tuning Findings

Two completed local ablation runs explored training length and negative sampling difficulty for Mini-DLRM and Hybrid Mini-DLRM + Semantic across seeds 42, 123, and 2026. These reporting cells use the completed summary values directly and do not rerun training.

The recommended setting from this pass is `Hybrid Mini-DLRM + Semantic`, `epochs=5`, `lr=1e-3`, `emb_dim=32`, and `train_negatives=12`.

In [ ]:
epoch_tuning = pd.DataFrame(
    [
        {"Model": "Hybrid Mini-DLRM + Semantic", "epochs": 5, "Recall@10_mean": 0.247196, "Recall@10_std": 0.004858, "NDCG@10_mean": 0.131841, "NDCG@10_std": 0.003375},
        {"Model": "Hybrid Mini-DLRM + Semantic", "epochs": 10, "Recall@10_mean": 0.239183, "Recall@10_std": 0.016698, "NDCG@10_mean": 0.125400, "NDCG@10_std": 0.006252},
        {"Model": "Mini-DLRM", "epochs": 5, "Recall@10_mean": 0.226362, "Recall@10_std": 0.011674, "NDCG@10_mean": 0.125478, "NDCG@10_std": 0.005522},
        {"Model": "Mini-DLRM", "epochs": 10, "Recall@10_mean": 0.233574, "Recall@10_std": 0.004858, "NDCG@10_mean": 0.126272, "NDCG@10_std": 0.006847},
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for model_name, group in epoch_tuning.groupby("Model"):
    group = group.sort_values("epochs")
    color = "#2563EB" if "Hybrid" in model_name else "#6B7280"
    axes[0].errorbar(group["epochs"], group["Recall@10_mean"], yerr=group["Recall@10_std"], marker="o", capsize=4, label=model_name, color=color)
    axes[1].errorbar(group["epochs"], group["NDCG@10_mean"], yerr=group["NDCG@10_std"], marker="o", capsize=4, label=model_name, color=color)
axes[0].set_title("Recall@10 vs epochs")
axes[1].set_title("NDCG@10 vs epochs")
for axis in axes:
    axis.set_xlabel("epochs")
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Recall@10 mean")
axes[1].set_ylabel("NDCG@10 mean")
axes[1].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

display(epoch_tuning)

Epoch tuning interpretation: Hybrid performs best at 5 epochs. Extending Hybrid to 10 epochs reduces mean Recall/NDCG and increases variance, suggesting semantic priors help early but longer training may overfit sparse collaborative signals or increase head/popularity bias. Mini-DLRM improves slightly from 5 to 10 epochs, which is consistent with pure collaborative models needing longer training to fit sparse interaction patterns.

In [ ]:
negative_tuning = pd.DataFrame(
    [
        {"Model": "Hybrid Mini-DLRM + Semantic", "train_negatives": 8, "Recall@10_mean": 0.257612, "Recall@10_std": 0.006620, "NDCG@10_mean": 0.141072, "NDCG@10_std": 0.003672},
        {"Model": "Hybrid Mini-DLRM + Semantic", "train_negatives": 12, "Recall@10_mean": 0.259615, "Recall@10_std": 0.007311, "NDCG@10_mean": 0.143954, "NDCG@10_std": 0.000702},
        {"Model": "Hybrid Mini-DLRM + Semantic", "train_negatives": 16, "Recall@10_mean": 0.260417, "Recall@10_std": 0.005004, "NDCG@10_mean": 0.142928, "NDCG@10_std": 0.003477},
        {"Model": "Mini-DLRM", "train_negatives": 8, "Recall@10_mean": 0.244391, "Recall@10_std": 0.021812, "NDCG@10_mean": 0.136589, "NDCG@10_std": 0.009900},
        {"Model": "Mini-DLRM", "train_negatives": 12, "Recall@10_mean": 0.248798, "Recall@10_std": 0.015625, "NDCG@10_mean": 0.139045, "NDCG@10_std": 0.009865},
        {"Model": "Mini-DLRM", "train_negatives": 16, "Recall@10_mean": 0.259215, "Recall@10_std": 0.016318, "NDCG@10_mean": 0.144534, "NDCG@10_std": 0.010475},
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for model_name, group in negative_tuning.groupby("Model"):
    group = group.sort_values("train_negatives")
    color = "#2563EB" if "Hybrid" in model_name else "#6B7280"
    axes[0].errorbar(group["train_negatives"], group["Recall@10_mean"], yerr=group["Recall@10_std"], marker="o", capsize=4, label=model_name, color=color)
    axes[1].errorbar(group["train_negatives"], group["NDCG@10_mean"], yerr=group["NDCG@10_std"], marker="o", capsize=4, label=model_name, color=color)
axes[0].set_title("Recall@10 vs train negatives")
axes[1].set_title("NDCG@10 vs train negatives")
for axis in axes:
    axis.set_xlabel("train negatives")
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Recall@10 mean")
axes[1].set_ylabel("NDCG@10 mean")
axes[1].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

display(negative_tuning)

In [ ]:
stability_rows = []
epoch_best = epoch_tuning[epoch_tuning["epochs"] == 5].copy()
epoch_best["Setting"] = "epochs=5, negatives=4"
negative_best = negative_tuning[negative_tuning["train_negatives"] == 12].copy()
negative_best["Setting"] = "epochs=5, negatives=12"
for frame in [epoch_best, negative_best]:
    stability_rows.append(frame[["Setting", "Model", "Recall@10_std", "NDCG@10_std"]])
stability = pd.concat(stability_rows, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for idx, metric in enumerate(["Recall@10_std", "NDCG@10_std"]):
    pivot = stability.pivot(index="Setting", columns="Model", values=metric)
    pivot.plot(kind="bar", ax=axes[idx], color=["#2563EB" if "Hybrid" in col else "#6B7280" for col in pivot.columns], rot=20)
    axes[idx].set_title(metric.replace("_", " "))
    axes[idx].set_ylabel("std across seeds")
    axes[idx].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(stability)

Negative-sampling interpretation: increasing train negatives from 4 to 8/12/16 improves ranking quality. Hybrid remains consistently more stable across seeds than Mini-DLRM. The best balanced hybrid setting is around `train_negatives=12`: strong Recall@10, best NDCG@10, and very low NDCG variance. At 16 negatives, Mini-DLRM nearly catches up in mean performance, but it still shows substantially higher variance.

## Long-tail Analysis Findings

The head/tail split shows that DLRM-style interaction models can become strongly head-biased. Mini-DLRM ranks head items well, but its tail Recall@10 drops sharply. Adding frozen semantic item embeddings improves tail Recall@10 and NDCG@10 relative to Mini-DLRM, indicating that product text can supply useful structure when interaction counts are low.

The long-tail result does not mean the semantic model solves tail recommendation outright. Tail performance remains much lower than head performance, which is expected in a sparse sampled-candidate setup with random negatives and limited metadata features.

## Hyperparameter Observations

The initial controlled ablation at `epochs=5`, `lr=1e-3`, `emb_dim=32`, and `train_negatives=4` showed Hybrid Mini-DLRM + Semantic with Recall@10 mean around 0.2472 versus Mini-DLRM around 0.2264. Hybrid also had lower Recall@10 variance across seeds in this setting.

The tuning pass sharpens that conclusion: Hybrid works best with moderate training length and harder negative sampling. The current recommended configuration is `Hybrid Mini-DLRM + Semantic`, `epochs=5`, `lr=1e-3`, `emb_dim=32`, and `train_negatives=12`. Semantic embeddings improve sparse recommendation not only by improving mean ranking quality, but also by reducing seed sensitivity and stabilizing optimization under harder negative sampling.

## Final Consolidated Findings

The final experimental setting is an Amazon Reviews 2023 Movies_and_TV subset with 832 users, 2,215 items, 7,356 interactions, and 0.9960 sparsity. Evaluation uses leave-last-out ranking with one held-out positive item and 99 sampled negatives per user. Because each test case has exactly one positive item, HitRate@10 and Recall@10 are effectively identical in this setup.

The project evolved through five stages. First, popularity, Two-Tower, and Mini-DLRM established collaborative baselines. On the initial tiny dense 47-item subset, Two-Tower and Mini-DLRM were similar, and semantic priors had little room to help. Second, expanding to a much larger sparse long-tail subset made tail behavior central, and semantic hybrid recommendation began outperforming purely collaborative models. Third, harder negative sampling improved ranking quality, especially NDCG, while semantic hybrid models stayed more stable across random seeds. Fourth, embedding dimension tuning exposed a capacity/stability tradeoff: larger collaborative embeddings increased capacity but could increase variance or overfit sparse interactions. Fifth, semantic encoder comparison showed that MiniLM, BGE-small, and e5-small behave differently as recommender priors.

The main conclusion is that semantic item embeddings become increasingly valuable under sparse long-tail recommendation settings. They improve tail-item discrimination, reduce seed sensitivity, and provide a stabilizing inductive bias when negative sampling becomes harder. The strongest general retrieval encoder is not automatically the best recommender semantic prior; alignment between semantic embedding geometry and recommender optimization matters.

## Final Benchmark Summary

| Model | Best observed setting | Recall@10 | HitRate@10 | NDCG@10 |
|---|---|---:|---:|---:|
| Popularity | sparse long-tail baseline | 0.2139 | 0.2139 | 0.1136 |
| Two-Tower | sparse long-tail baseline | 0.1094 | 0.1094 | 0.0479 |
| Mini-DLRM | tuned negatives=16 | 0.2592 | 0.2592 | 0.1445 |
| Hybrid MiniLM | epochs=5, emb_dim=32, negatives=12 | 0.2596 | 0.2596 | 0.1440 |
| Hybrid BGE | epochs=8, emb_dim=64, negatives=12 | ~0.275 | ~0.275 | ~0.137 |

The best balanced/stable configuration is Hybrid Mini-DLRM + Semantic with `sentence-transformers/all-MiniLM-L6-v2`, `epochs=5`, `lr=1e-3`, `emb_dim=32`, and `train_negatives=12`. The best retrieval-oriented configuration is Hybrid Mini-DLRM + Semantic with `BAAI/bge-small-en-v1.5`, `epochs=8`, `lr=1e-3`, `emb_dim=64`, and `train_negatives=12`, reaching Recall@10 mean around 0.275 and NDCG@10 mean around 0.137.

BGE required higher projection capacity and longer optimization to transfer retrieval-oriented semantic quality into the recommender objective. MiniLM remained the stronger balanced/stable baseline, especially when low variance across seeds mattered.

## Lessons Learned / Key Insights

- Semantic embeddings help more in sparse long-tail settings than in tiny dense item universes.
- Semantic priors improve optimization stability and tail-item discrimination.
- Harder negative sampling improves ranking quality, especially NDCG.
- Semantic encoder choice materially affects hybrid recommender performance.
- Encoder-specific tuning matters; retrieval quality does not automatically translate into recommendation quality.
- Sparse recommendation exhibits a capacity/stability tradeoff, so larger embeddings are not always better.

Most important insight: the project evolved from a simple "DLRM + semantic embeddings" implementation into a systematic analysis of semantic priors under sparse long-tail recommendation optimization.

## Limitations

- Evaluation uses sampled candidates, not full-catalog retrieval.
- Random negatives are simple and may not reflect hard production ranking cases.
- The project does not include sequential user modeling.
- The project does not include graph-based collaborative modeling.
- The project does not separate retrieval and reranking stages.
- Encoder-specific tuning was informative but not exhaustive.
- The semantic encoder is frozen, so it cannot adapt its text space to recommendation-specific preference signals.
- Metadata quality varies by item; missing or generic descriptions reduce the value of semantic features.
- The experiment is intentionally compact and should be treated as directional rather than a production benchmark.

## Key Takeaways

- Semantic embeddings help more in sparse long-tail settings than in tiny dense item universes.
- DLRM-style interaction models can be strongly head-biased.
- Semantic priors can improve stability and generalization across seeds.
- Harder negative sampling improves ranking quality, with `train_negatives=12` the best balanced hybrid setting observed.
- Recommendation quality depends heavily on dataset regime, metadata coverage, negative sampling, and feature richness.

The staged experiment separates popularity, collaborative ID embeddings, feature interaction, and semantic content signals. The hybrid ranker is designed to retain collaborative behavior while giving long-tail and cold-start items a content-based representation before they accumulate many interactions.

## Future Work

- hard negative mining
- popularity-aware negative sampling
- ANN retrieval with Faiss
- SASRec or BERT4Rec sequence modeling
- LLM reranking
- multimodal product embeddings
- fine-tuning semantic encoders on recommendation pairs

## References and Resources

- [Amazon Reviews 2023 dataset](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023), McAuley Lab
- DLRM: deep learning recommendation models with sparse embeddings, dense features, and feature interactions
- Semantic embedding models: MiniLM, e5, and BGE sentence embedding families
- Long-tail recommendation motivation: ranking remains hardest when item feedback is sparse and popularity is highly skewed
- [Negative Sampling in Recommendation: A Survey and Future Directions](https://dl.acm.org/doi/10.1145/3793855), ACM Digital Library